silver notebook _ Anannya Dash 23053026

In [0]:
bronze=spark.read.csv("/Volumes/workspace/job_skills/raw_files/raw/postings",header=True)


In [0]:
unique_postings = bronze.select(
    "posting_id"
).distinct().count()

print("Unique posting IDs:", unique_postings)

Unique posting IDs: 54000


In [0]:
bronze_path = "/Volumes/workspace/job_skills/raw_files/bronze_postings"

bronze_df = spark.read.option("header", True).csv(
    "/Volumes/workspace/job_skills/raw_files/raw/postings"
)

bronze_df.write.format("delta").mode("overwrite").save(bronze_path)

spark.sql("""
CREATE TABLE IF NOT EXISTS job_skills.bronze_postings
USING DELTA
AS SELECT * FROM delta.`/Volumes/workspace/job_skills/raw_files/bronze_postings`
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
deduped = bronze.dropDuplicates(
    ["posting_id"]
)

In [0]:
deduped_count = deduped.count()

print("After deduplication:", deduped_count)

After deduplication: 54000


In [0]:
assert deduped_count == 54000, (
    f"Expected 54,000 rows after deduplication, "
    f"got {deduped_count}"
)

In [0]:
duplicates_removed = bronze.count() - deduped_count

print("Duplicates removed:", duplicates_removed)

assert duplicates_removed == 1350

Duplicates removed: 1350


performed posting-level deduplication before skill explosion to prevent duplicate postings from inflating skill frequencie

In [0]:
companies = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv("/Volumes/workspace/job_skills/raw_files/raw/companies")
)

In [0]:
display(companies.limit(10))


company_id,company_name,sector,city
CMP050,Company 051,IT Services,Bengaluru
CMP051,Company 052,IT Services,Hyderabad
CMP052,Company 053,IT Services,Pune
CMP053,Company 054,IT Services,Chennai
CMP054,Company 055,IT Services,Mumbai
CMP055,Company 056,IT Services,Delhi NCR
CMP056,Company 057,IT Services,Noida
CMP057,Company 058,IT Services,Kolkata
CMP058,Company 059,IT Services,Ahmedabad
CMP059,Company 060,IT Services,Kochi


In [0]:
print(companies.count())

400


In [0]:
companies.select(
    "company_id"
).distinct().count()

400

In [0]:
from pyspark.sql import functions as F

In [0]:
validated = deduped.withColumn(
    "is_invalid_skill",
    (
        F.col("skills_raw").isNull()
        |
        (F.trim(F.col("skills_raw")) == "")
        |
        (
            F.lower(
                F.trim(F.col("skills_raw"))
            ) == "not specified"
        )
    )
)

In [0]:
validated.groupBy(
    "is_invalid_skill"
).count().show()

+----------------+-----+
|is_invalid_skill|count|
+----------------+-----+
|           false|52650|
|            true| 1350|
+----------------+-----+



In [0]:
invalid_skill_count = validated.filter(
    F.col("is_invalid_skill")
).count()

print(
    "No-skill / Not-specified:",
    invalid_skill_count
)
assert invalid_skill_count == 1350

No-skill / Not-specified: 1350


In [0]:
valid_company_ids = companies.select(
    "company_id"
).distinct()

In [0]:
validated = (
    validated
    .join(
        valid_company_ids.withColumn(
            "company_exists",
            F.lit(True)
        ),
        on="company_id",
        how="left"
    )
)

In [0]:
validated = validated.withColumn(
    "is_unknown_company",
    F.col("company_exists").isNull()
)

In [0]:
unknown_company_count = validated.filter(
    F.col("is_unknown_company")
).count()

print(
    "Unknown company:",
    unknown_company_count
)



Unknown company: 540


In [0]:
assert unknown_company_count==540

In [0]:
display(
    validated
    .filter(F.col("is_unknown_company"))
    .select(
        "posting_id",
        "company_id"
    )
    .limit(20)
)

posting_id,company_id
JP029779,CMP999
JP029794,CMP999
JP032795,CMP999
JP032804,CMP999
JP041780,CMP999
JP041784,CMP999
JP044794,CMP999
JP044796,CMP999
JP035775,CMP999
JP035784,CMP999


In [0]:
validated = validated.withColumn(
    "is_invalid_date",
    (
        F.col("posted_date").isNull()
        |
        (F.col("posted_date") == "0000-00-00")
    )
)

In [0]:
invalid_date_count = validated.filter(
    F.col("is_invalid_date")
).count()

print(
    "Invalid dates:",
    invalid_date_count
)

Invalid dates: 270


In [0]:
assert invalid_date_count == 270

In [0]:
display(
    validated
    .filter(F.col("is_invalid_date"))
    .select(
        "posting_id",
        "posted_date"
    )
    .limit(20)
)

posting_id,posted_date
JP032816,0000-00-00
JP041809,0000-00-00
JP044805,0000-00-00
JP044808,0000-00-00
JP044816,0000-00-00
JP035811,0000-00-00
JP035818,0000-00-00
JP035819,0000-00-00
JP038816,0000-00-00
JP047812,0000-00-00


In [0]:
validated = validated.withColumn(
    "reject_reason",
    F.when(
        F.col("is_invalid_skill"),
        F.lit("no_skills_or_not_specified")
    )
    .when(
        F.col("is_unknown_company"),
        F.lit("unknown_company")
    )
    .when(
        F.col("is_invalid_date"),
        F.lit("invalid_date")
    )
    .otherwise(
        F.lit(None)
    )
)

In [0]:
display(
    validated.groupBy(
        "reject_reason"
    ).count()
)

reject_reason,count
null,51840
no_skills_or_not_specified,1350
unknown_company,540
invalid_date,270


In [0]:
no_skill_count = validated.filter(
    F.col("reject_reason") ==
    "no_skills_or_not_specified"
).count()

unknown_company_count = validated.filter(
    F.col("reject_reason") ==
    "unknown_company"
).count()

invalid_date_count = validated.filter(
    F.col("reject_reason") ==
    "invalid_date"
).count()

print("No skills:", no_skill_count)
print("Unknown company:", unknown_company_count)
print("Invalid date:", invalid_date_count)

No skills: 1350
Unknown company: 540
Invalid date: 270


In [0]:
rejected_count = validated.filter(
    F.col("reject_reason").isNotNull()
).count()

print("Total rejected:", rejected_count)

Total rejected: 2160


In [0]:
valid_postings = validated.filter(
    F.col("reject_reason").isNull()
)

In [0]:
valid_count = valid_postings.count()

print("Valid postings:", valid_count)

Valid postings: 51840


In [0]:
assert valid_count == 51840

In [0]:
rejected_postings = validated.filter(
    F.col("reject_reason").isNotNull()
)

In [0]:
print(
    "Rejected postings:",
    rejected_postings.count()
)

Rejected postings: 2160


In [0]:
valid_postings = validated.filter(
    F.col("reject_reason").isNull()
)

In [0]:
valid_count = valid_postings.count()

print("Valid postings:", valid_count)

Valid postings: 51840


In [0]:
assert valid_count == 51840

In [0]:
rejected_postings = validated.filter(
    F.col("reject_reason").isNotNull()
)

In [0]:
print(
    "Rejected postings:",
    rejected_postings.count()
)

Rejected postings: 2160


In [0]:
rejected_postings.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/job_skills/raw_files/raw/silver/rejected_postings"
    )

In [0]:
valid_postings.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/job_skills/raw_files/raw/silver/valid_postings"
    )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.job_skills.silver_rejected_postings
USING DELTA
AS SELECT * FROM delta.`/Volumes/workspace/job_skills/raw_files/raw/silver/rejected_postings`

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.job_skills.silver_valid_postings
USING DELTA
AS SELECT * FROM delta.`/Volumes/workspace/job_skills/raw_files/raw/silver/valid_postings`

num_affected_rows,num_inserted_rows


In [0]:
valid_postings = spark.table(
    "workspace.job_skills.silver_valid_postings"
)

In [0]:
posting_skill = valid_postings.withColumn(
    "skill_array",
    F.split(
        F.col("skills_raw"),
        r"[,;:]"
    )
)

In [0]:
display(
    posting_skill.select(
        "posting_id",
        "skills_raw",
        "skill_array"
    ).limit(20)
)

posting_id,skills_raw,skill_array
JP040516,Kafka;PowerBI;Docker;Databricks,"List(Kafka, PowerBI, Docker, Databricks)"
JP040546,Spark;AWS;PowerBI;Docker;Databricks,"List(Spark, AWS, PowerBI, Docker, Databricks)"
JP040579,Airflow;GCP;PowerBI;Docker;Databricks,"List(Airflow, GCP, PowerBI, Docker, Databricks)"
JP040598,AWS;Docker;PowerBI;Databricks,"List(AWS, Docker, PowerBI, Databricks)"
JP040608,Python;Kubernetes;PowerBI;Docker;Databricks,"List(Python, Kubernetes, PowerBI, Docker, Databricks)"
JP040614,AWS;Kubernetes;PowerBI;Docker;Databricks,"List(AWS, Kubernetes, PowerBI, Docker, Databricks)"
JP040615,Azure;Kubernetes;PowerBI;Docker;Databricks,"List(Azure, Kubernetes, PowerBI, Docker, Databricks)"
JP040619,Terraform ; Kubernetes ; PowerBI ; Docker ; Databricks,"List( Terraform , Kubernetes , PowerBI , Docker , Databricks )"
JP040629,Hive;Terraform;PowerBI;Docker;Databricks,"List(Hive, Terraform, PowerBI, Docker, Databricks)"
JP040645,Hive;Scala;PowerBI;Docker,"List(Hive, Scala, PowerBI, Docker)"


In [0]:
posting_skill = posting_skill.withColumn(
    "skill_raw",
    F.explode(
        F.col("skill_array")
    )
)

In [0]:
display(
    posting_skill.select(
        "posting_id",
        "skills_raw",
        "skill_raw"
    ).limit(30)
)

posting_id,skills_raw,skill_raw
JP040516,Kafka;PowerBI;Docker;Databricks,Kafka
JP040516,Kafka;PowerBI;Docker;Databricks,PowerBI
JP040516,Kafka;PowerBI;Docker;Databricks,Docker
JP040516,Kafka;PowerBI;Docker;Databricks,Databricks
JP040546,Spark;AWS;PowerBI;Docker;Databricks,Spark
JP040546,Spark;AWS;PowerBI;Docker;Databricks,AWS
JP040546,Spark;AWS;PowerBI;Docker;Databricks,PowerBI
JP040546,Spark;AWS;PowerBI;Docker;Databricks,Docker
JP040546,Spark;AWS;PowerBI;Docker;Databricks,Databricks
JP040579,Airflow;GCP;PowerBI;Docker;Databricks,Airflow


In [0]:
posting_skill = posting_skill.withColumn(
    "skill",
    F.lower(
        F.trim(
            F.col("skill_raw")
        )
    )
)

In [0]:
display(
    posting_skill.select(
        "posting_id",
        "skills_raw",
        "skill_raw","skill"
    ).limit(30)
)

posting_id,skills_raw,skill_raw,skill
JP040516,Kafka;PowerBI;Docker;Databricks,Kafka,kafka
JP040516,Kafka;PowerBI;Docker;Databricks,PowerBI,powerbi
JP040516,Kafka;PowerBI;Docker;Databricks,Docker,docker
JP040516,Kafka;PowerBI;Docker;Databricks,Databricks,databricks
JP040546,Spark;AWS;PowerBI;Docker;Databricks,Spark,spark
JP040546,Spark;AWS;PowerBI;Docker;Databricks,AWS,aws
JP040546,Spark;AWS;PowerBI;Docker;Databricks,PowerBI,powerbi
JP040546,Spark;AWS;PowerBI;Docker;Databricks,Docker,docker
JP040546,Spark;AWS;PowerBI;Docker;Databricks,Databricks,databricks
JP040579,Airflow;GCP;PowerBI;Docker;Databricks,Airflow,airflow


In [0]:
raw_skill_count = posting_skill.select(
    "skill_raw"
).distinct().count()

print(
    "Distinct raw skills:",
    raw_skill_count
)

Distinct raw skills: 57


In [0]:
clean_skill_count = posting_skill.select(
    "skill"
).distinct().count()

print(
    "Distinct normalized skills:",
    clean_skill_count
)

Distinct normalized skills: 20


In [0]:
assert raw_skill_count == 57
assert clean_skill_count == 20

In [0]:
display(
    posting_skill
    .select("skill")
    .distinct()
    .orderBy("skill")
)

skill
airflow
aws
azure
databricks
dbt
docker
gcp
hadoop
hive
java


In [0]:
empty_after_explode = posting_skill.filter(
    F.col("skill") == ""
).count()

print(
    "Empty skills after explosion:",
    empty_after_explode
)

Empty skills after explosion: 0


In [0]:
assert empty_after_explode == 0

In [0]:
posting_skill.printSchema()

root
 |-- company_id: string (nullable = true)
 |-- posting_id: string (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- skills_raw: string (nullable = true)
 |-- is_invalid_skill: boolean (nullable = true)
 |-- company_exists: boolean (nullable = true)
 |-- is_unknown_company: boolean (nullable = true)
 |-- is_invalid_date: boolean (nullable = true)
 |-- reject_reason: string (nullable = true)
 |-- skill_array: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- skill_raw: string (nullable = false)
 |-- skill: string (nullable = false)



In [0]:
silver_posting_skill = posting_skill.select(
    "posting_id",
    "company_id",
    "posted_date",
    "skill_raw",
    "skill"
)

In [0]:
silver_posting_skill.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/job_skills/raw_files/silver/posting_skill"
    )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.job_skills.silver_posting_skill
USING DELTA
AS SELECT * FROM delta.`/Volumes/workspace/job_skills/raw_files/silver/posting_skill`

num_affected_rows,num_inserted_rows


In [0]:
print("========== SILVER VALIDATION ==========")

print(
    "Bronze rows:",
    bronze.count()
)

print(
    "Unique postings:",
    deduped.count()
)

print(
    "Duplicates removed:",
    bronze.count() - deduped.count()
)

print(
    "No skills:",
    no_skill_count
)

print(
    "Unknown company:",
    unknown_company_count
)

print(
    "Invalid date:",
    invalid_date_count
)

print(
    "Valid postings:",
    valid_count
)

print(
    "Raw skill strings:",
    raw_skill_count
)

print(
    "Normalized skills:",
    clean_skill_count
)

print(
    "Empty normalized skills:",
    empty_after_explode
)

========== SILVER VALIDATION ==========
Bronze rows: 55350
Unique postings: 54000
Duplicates removed: 1350
No skills: 1350
Unknown company: 540
Invalid date: 270
Valid postings: 51840
Raw skill strings: 57
Normalized skills: 20
Empty normalized skills: 0


In [0]:
duplicate_skill_rows = (
    silver_posting_skill
    .groupBy(
        "posting_id",
        "skill"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

display(duplicate_skill_rows.count())

0